In [0]:
# Install required libraries for this notebook
%pip install xgboost shap

In [0]:
# Notebook 5: Order Quality Risk Scoring
# Food Delivery Analysis
# -----------------------------------------

from pyspark.sql.functions import col
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Load order features from Parquet
order_features = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/order_features.parquet"
)

print(f"Loaded {order_features.count():,} orders with {len(order_features.columns)} features.")
order_features.display()

print(f"Total orders: {order_features.count():,}")
print(f"Columns: {order_features.columns}")

In [0]:
# Convert to Pandas for modelling
order_pd = order_features.toPandas()

# Create binary risk label
# High risk = order_quality_risk_score above 0.5
# Low risk = order_quality_risk_score 0.5 and below
order_pd["high_risk"] = (order_pd["order_quality_risk_score"] > 0.5).astype(int)

# Drop columns not needed for modelling
order_pd = order_pd.drop(columns=[
    "order_id", "user_id", "restaurant_id",
    "order_status", "order_quality_risk_score"
])

# Encode text columns
le = LabelEncoder()
for col_name in ["cuisine", "city", "area", "traffic_level",
                 "driver_vehicle", "driver_availability", "payment_method"]:
    order_pd[col_name] = le.fit_transform(order_pd[col_name])

# Separate features and target
X = order_pd.drop(columns=["high_risk"])
y = order_pd["high_risk"]

print(f"Features shape: {X.shape}")
print(f"High risk rate: {y.mean():.1%}")
print(f"\nFeature columns:\n{list(X.columns)}")

In [0]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} orders")
print(f"Test set: {X_test.shape[0]:,} orders")
print(f"\nHigh risk rate in training set: {y_train.mean():.1%}")
print(f"High risk rate in test set: {y_test.mean():.1%}")

In [0]:
# Train XGBoost order quality risk model
with mlflow.start_run(run_name="XGBoost Order Quality Risk"):

    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    )

    xgb_model.fit(X_train, y_train)

    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.xgboost.log_model(xgb_model, "xgboost_order_risk_model")

    print(f"Accuracy:  {accuracy:.1%}")
    print(f"ROC AUC:   {roc_auc:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred,
          target_names=["Low Risk", "High Risk"]))